In [1]:
import os
import re
import json
from datetime import datetime
import rasterio
import geopandas as gpd
from rasterio.warp import calculate_default_transform, reproject, Resampling
from pathlib import Path
import shutil
from shapely.geometry import box

# Renaming of the file in the format (SS_SR_YYYYMMDDUTC_UU_BM.tif)

In [64]:
#Input the file
input_path='/path/to/input_file/input.tif'
output_path_5070="/path/to/output_path_5070/Reprojected_BM_5070.tif"
output_path_4326="/path/to/output_path_4326/Reprojected_BM_4326.tif"
target_epsg=5070                       #user input projected coordinate system
target_epsg2=4326

# Step 1: Extract original projection
with rasterio.open(input_path) as src:
    original_crs = src.crs

def reproject_raster(input_file, output_file, target_epsg):
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    with rasterio.open(input_file) as src:
        transform, width, height = calculate_default_transform(
            src.crs, f"EPSG:{target_epsg}", src.width, src.height, *src.bounds)

        metadata = src.meta.copy()
        metadata.update({
            'crs': f"EPSG:{target_epsg}",
            'transform': transform,
            'width': width,
            'height': height,
            'compress': 'lzw'
        })

        with rasterio.open(output_file, 'w', **metadata) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=f"EPSG:{target_epsg}",
                    resampling=Resampling.nearest
                )

    print(f" Reprojected file saved to: {output_file}")


# Run reprojections
reproject_raster(input_path, output_path_5070, 5070)
reproject_raster(input_path, output_path_4326, 4326)



 Reprojected file saved to: /Users/ddevi/Library/CloudStorage/OneDrive-TheUniversityofAlabama/FIM Repository/All_filtered/Output-3classFIM-2/Reprojected_BM_5070.tif
 Reprojected file saved to: /Users/ddevi/Library/CloudStorage/OneDrive-TheUniversityofAlabama/FIM Repository/All_filtered/Output-3classFIM-2/Reprojected_BM_4326.tif


## Get the resolution from the Projected Raster File (SR)

In [65]:
with rasterio.open(output_path_5070) as src:
    res_x, res_y = src.res  # src.res returns (pixel_width, pixel_height)
# Use average if they're slightly different due to reprojection
    avg_res = (res_x + res_y) / 2
    #res_x=round(res_x)

# Round to 1 decimal place
    rounded_res = round(avg_res, 1)  # e.g., 0.28 -> 0.3, 9.99 -> 10.0

    # Replace '.' with '_' and add 'm' suffix
    res_str = f"{str(rounded_res).replace('.', '_')}m"

print("Using resolution:", res_str)

Using resolution: 10_2m


## Get the dms code for the centroid of the raster file (UU)

In [66]:

def get_raster_centroid(output_path_4326):
    with rasterio.open(output_path_4326) as src:
        bounds = src.bounds
        centroid_x = (bounds.left + bounds.right) / 2
        centroid_y = (bounds.top + bounds.bottom) / 2
        crs = src.crs
    return centroid_x, centroid_y, crs

x, y, crs = get_raster_centroid(output_path_4326)

def decimal_to_dms_str(dec, is_lat=True):
    direction = ''
    if is_lat:
        direction = 'N' if dec >= 0 else 'S'
    else:
        direction = 'E' if dec >= 0 else 'W'

    dec = abs(dec)
    degrees = int(dec)
    minutes = int((dec - degrees) * 60)
    seconds = int(((dec - degrees) * 60 - minutes) * 60)

    return f"{degrees:02d}{minutes:02d}{seconds:02d}{direction}"

# Convert both
lon_str = decimal_to_dms_str(x, is_lat=False)
lat_str = decimal_to_dms_str(y, is_lat=True)

# Concatenate
dms_code = lon_str + lat_str
print(dms_code)


780123W352131N


## Rename the benchmark file and store in the repository

In [67]:
#Input the naming convention
sensor_code = input("Enter the sensor name code (e.g.,PSS;BLE;S1A): ")
res=res_str
flood_date = input("Enter the date of flood in YYYYMMDDT format (e.g., 20230615T)/Enter Synthetic Flood Value (e.g.,50,100,500): ")
unique_identifier=dms_code
#suffix=BM
# Combine flood date and dms_code for folder name
folder_name = f"{flood_date}_{unique_identifier}"
base_destination=Path("/path/to/store/output_file/Level_2")
destination_folder = base_destination / folder_name
destination_folder.mkdir(parents=True, exist_ok=True)
# Input TIF file
#original_path = Path("/Users/ddevi/Library/CloudStorage/OneDrive-TheUniversityofAlabama/FIM Repository/BLE/Test_BLE/11010004/100yr/ble_huc_08040203_extent_100yr.tif") # replace with your actual file path
new_filename = f"{sensor_code}_{res}_{flood_date}_{unique_identifier}_{'BM'}.tif"
new_path = destination_folder / new_filename

# Rename the file
#original_path.rename(new_path)
# Step 4: Copy instead of rename
shutil.copy2(output_path_4326, new_path)
print(f" File copied and renamed to: {new_path}")
print(f"File renamed to: {new_path.name}")


Enter the sensor name code (e.g.,PSS;BLE;S1A):  PSS
Enter the date of flood in YYYYMMDDT format (e.g., 20230615T)/Enter Synthetic Flood Value (e.g.,50,100,500):  20161014


 File copied and renamed to: /Users/ddevi/Library/CloudStorage/OneDrive-TheUniversityofAlabama/FIM Repository/FIM_Database/Level_2/20161014_780123W352131N/PSS_10_2m_20161014_780123W352131N_BM.tif
File renamed to: PSS_10_2m_20161014_780123W352131N_BM.tif


## Get the auto metadata from the raster and enter manual metadata

In [68]:
with rasterio.open(output_path_4326) as src:
        metadata = {
            "File_Name": new_filename,
            "Format": src.driver,
            "Resolution": rounded_res,
            "Extent": {
                "xmin": src.bounds.left,
                "ymin": src.bounds.bottom,
                "xmax": src.bounds.right,
                "ymax": src.bounds.top
            },
            "DMS_Code_centroid":dms_code,
            "Projection": src.crs.to_string() if src.crs else "Unknown",
            #"Projection": original_crs,
            "Bands": src.count,
            "Rows": src.height,
            "Columns": src.width,
            # "Title": input("Enter Title: "),
            "State": input("Enter State: "),
            "Description": input("Enter Description: "),
            "River Name": input("Enter River Name: "),
            "Source": "SDML Lab, The University of Alabama, Tuscaloosa, USA",  # Fixed
            "Location of the centroid of the flood map" : (x,y) ,
            "Date of the Flood" : input("Enter Date of Flood / Synthetic Flooding Event: "),
            "Keywords": ["flood", "hazard", "simulation", "GIS"],
            "Access_Rights": "Public",
            "Level": input("Enter Level (e.g., Level_1): ")
              
        }

Enter State:  North Carolina, USA
Enter Description:  FIM generated from Planet’s Imagery of 2016 Hurricane Matthew with a spatial resolution of 10m.
Enter River Name:  Neuse River
Enter Date of Flood / Synthetic Flooding Event:  20161014
Enter Level (e.g., Level_1):  Level_2


## Save the json file

In [69]:
metadata_filename = new_path.name.replace('BM.tif', 'metadata.json')
metadata_path = new_path.parent / metadata_filename

with open(metadata_path, 'w') as json_file:
    json.dump(metadata, json_file, indent=4)

print(f"📝 Metadata saved to: {metadata_path}")

📝 Metadata saved to: /Users/ddevi/Library/CloudStorage/OneDrive-TheUniversityofAlabama/FIM Repository/FIM_Database/Level_2/20161014_780123W352131N/PSS_10_2m_20161014_780123W352131N_metadata.json


In [70]:
# Define the GeoPackage path in the same folder as the TIF
gpkg_path = new_path.parent / new_path.name.replace('BM.tif', 'AOI.gpkg')

# Generate bounding box
with rasterio.open(output_path_5070) as src:
    bbox = box(*src.bounds)
    crs = src.crs

# Create GeoDataFrame with metadata and geometry
gdf = gpd.GeoDataFrame([metadata], geometry=[bbox], crs=crs)

# Save to GeoPackage
gdf.to_file(gpkg_path, layer='AOI', driver="GPKG")


print(f"✅ Bounding box saved as GeoPackage: {gpkg_path}")

✅ Bounding box saved as GeoPackage: /Users/ddevi/Library/CloudStorage/OneDrive-TheUniversityofAlabama/FIM Repository/FIM_Database/Level_2/20161014_780123W352131N/PSS_10_2m_20161014_780123W352131N_AOI.gpkg


In [61]:
pip install awscli

  Using cached docutils-0.19-py3-none-any.whl.metadata (2.7 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
  Using cached rsa-4.7.2-py3-none-any.whl.metadata (3.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 71.4 MB/s eta 0:00:00
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached docutils-0.19-py3-none-any.whl (570 kB)
Using cached rsa-4.7.2-py3-none-any.whl (34 kB)
  Attempting uninstall: rsa
    Found existing installation: rsa 4.9.1
    Uninstalling rsa-4.9.1:
      Successfully uninstalled rsa-4.9.1
  Attempting uninstall: botocore90m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [docutils]
    Found existing installation: botocore 1.38.22━━━━━━━━━━━━━ 1/5 [docutils]
    Uninstalling botocore-1.38.22:0m╺━━━━━━━━━━━━━━━ 3/5 [botocore]
      Successfully uninstalled botocore-1.38.22━━━━━━━━━━━━━━━ 3/5 [botocore]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 

## AWS UPLOAD

In [2]:
!aws --version

aws-cli/1.40.41 Python/3.10.0 Darwin/24.5.0 botocore/1.38.42


In [1]:
import os

# Set your AWS credentials and region

os.environ['AWS_ACCESS_KEY_ID'] = '***********'
os.environ['AWS_SECRET_ACCESS_KEY'] = '**********'
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

# Verify that the environment variables are set correctly

print("AWS Access Key:", os.environ.get('AWS_ACCESS_KEY_ID'))
print("AWS Secret Access Key:", os.environ.get('AWS_SECRET_ACCESS_KEY'))
print("AWS Default Region:", os.environ.get('AWS_DEFAULT_REGION'))

AWS Access Key: ***********
AWS Secret Access Key: **********
AWS Default Region: us-east-1


In [13]:
import boto3
s3 = boto3.client('s3')
print(s3.list_buckets())

{'ResponseMetadata': {'RequestId': '2SQRV099W6GQ5J51', 'HostId': 'ganDXTvddKTf7L/RRryjhq1tseMVqntbAzVltfjdEAbir3VW3hO6UJWhjUBqAdAcsSI8DJE+ohALdfX0EV7KPDEgRsXt1acl', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'ganDXTvddKTf7L/RRryjhq1tseMVqntbAzVltfjdEAbir3VW3hO6UJWhjUBqAdAcsSI8DJE+ohALdfX0EV7KPDEgRsXt1acl', 'x-amz-request-id': '2SQRV099W6GQ5J51', 'date': 'Thu, 03 Jul 2025 17:34:11 GMT', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'Buckets': [{'Name': 'awi-website-backups', 'CreationDate': datetime.datetime(2024, 11, 26, 5, 55, 17, tzinfo=tzutc())}, {'Name': 'camels-nwm-reanalysis', 'CreationDate': datetime.datetime(2025, 6, 12, 23, 5, 2, tzinfo=tzutc())}, {'Name': 'ciroh-community-ngen-datastream-copy', 'CreationDate': datetime.datetime(2025, 4, 30, 4, 1, 30, tzinfo=tzutc())}, {'Name': 'ciroh-fim', 'CreationDate': datetime.datetime(2023, 5, 8, 17, 5, 38, tzinfo=tzutc())}, {'Name': 'ciroh-ngen-ngiab-hf', 'Creati